# Domain A: Information Extraction - Exploration

This notebook explores the 10 approaches for information extraction.

## Setup

In [ ]:
import sys
sys.path.append('..')

from src.domain_a_information_extraction.data_generator import create_ie_dataset, get_ie_fields
from src.domain_a_information_extraction.approach_01_rule_based import RuleBasedIE
from src.domain_a_information_extraction.approach_02_classical_ml import ClassicalMLIE
from src.core.metrics import compute_ie_metrics
import pandas as pd

## Generate Dataset

In [ ]:
dataset = create_ie_dataset(n_train=1000, n_val=200, n_test=200)
print(f"Train: {len(dataset['train']['X'])}")
print(f"Test: {len(dataset['test']['X'])}")
print(f"\nSample resume:\n{dataset['train']['X'][0][:500]}...")

## Compare Approaches

In [ ]:
results = []

# Rule-based
rule_model = RuleBasedIE()
rule_model.train(dataset['train']['X'], dataset['train']['y'])
rule_pred = rule_model.predict(dataset['test']['X'])
rule_metrics = compute_ie_metrics(dataset['test']['y'], rule_pred, get_ie_fields())
results.append({'Approach': 'Rule-Based', **rule_metrics})

# Classical ML
ml_model = ClassicalMLIE()
ml_model.train(dataset['train']['X'], dataset['train']['y'])
ml_pred = ml_model.predict(dataset['test']['X'])
ml_metrics = compute_ie_metrics(dataset['test']['y'], ml_pred, get_ie_fields())
results.append({'Approach': 'Classical ML', **ml_metrics})

pd.DataFrame(results)

## Analyze Failures

In [ ]:
failures = rule_model.collect_failure_cases(
    dataset['test']['X'], 
    dataset['test']['y'], 
    rule_pred, 
    n_cases=5
)

for f in failures:
    print(f"Index {f['index']}:")
    for field, mismatch in f['mismatches'].items():
        print(f"  {field}: expected '{mismatch['expected']}', got '{mismatch['predicted']}'")
    print()